# Dia4 - Gerador e Ingestão de Dados (Full Load, COPY INTO, Auto Loader, CDC)

**Objetivo:** Gerar dados sintéticos de pedidos e demonstrar 4 padrões de ingestão para a camada Bronze: Full Load, COPY INTO (idempotente), Auto Loader (contínuo) e CDC (Change Data Capture).

| Item | Valor |
|---|---|
| **Camada** | Bronze (múltiplas estratégias de ingestão) |
| **Entrada** | Arquivos JSON gerados sinteticamente em `/Volumes/{catalog}/landing/files/pedidos/` |
| **Saída** | `{catalog}.bronze.pedidos_full_load`, `.pedidos_copy_into`, `.pedidos_autoloader`, `.pedidos_cdc_raw` |
| **Ordem no pipeline** | Primeiro passo da camada Bronze para o dataset de pedidos |

**Resumo das tabelas criadas:**
- `pedidos_full_load` — Carga completa (overwrite), recarrega tudo sempre
- `pedidos_copy_into` — Carga incremental idempotente, não duplica ao reexecutar
- `pedidos_autoloader` — Ingestão contínua com checkpoint + evolução de schema
- `pedidos_cdc_raw` — Preserva histórico de operações (I/U/D) sem aplicar regras de negócio

In [0]:
dbutils.widgets.text("catalog", "capgemini_academy", "Nome do catálogo no Unity Catalog")
catalog = dbutils.widgets.get("catalog")

## Setup: imports, variáveis de caminho e criação de schemas/volumes

Configura todos os caminhos de Volume e nomes de tabela que serão usados. Cria schemas e volumes com `IF NOT EXISTS` para garantir idempotência.

In [0]:
import json
from datetime import datetime, timedelta
from pyspark.sql.functions import current_timestamp, col

catalog_name = catalog
landing_schema = "landing"
bronze_schema = "bronze"
landing_volume = "files"
metadata_volume = "metadata"

source_folder = "pedidos"
checkpoint_folder = "_checkpoint/pedidos_autoloader"
schema_folder = "_schema/pedidos_autoloader"

table_full = f"{catalog_name}.{bronze_schema}.pedidos_full_load"
table_copy = f"{catalog_name}.{bronze_schema}.pedidos_copy_into"
table_auto = f"{catalog_name}.{bronze_schema}.pedidos_autoloader"
table_cdc_raw = f"{catalog_name}.{bronze_schema}.pedidos_cdc_raw"

source_path = f"/Volumes/{catalog_name}/{landing_schema}/{landing_volume}/{source_folder}"
metadata_path = f"/Volumes/{catalog_name}/{landing_schema}/{metadata_volume}"
checkpoint_path = f"{metadata_path}/{checkpoint_folder}"
schema_path = f"{metadata_path}/{schema_folder}"

print("Source path:     ", source_path)
print("Metadata path:   ", metadata_path)
print("Checkpoint path: ", checkpoint_path)
print("Schema path:     ", schema_path)

In [0]:
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{landing_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{bronze_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{landing_schema}.{landing_volume}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{landing_schema}.{metadata_volume}")

dbutils.fs.mkdirs(source_path)
dbutils.fs.mkdirs(checkpoint_path)
dbutils.fs.mkdirs(schema_path)

## Funções utilitárias para geração de dados

Define `write_jsonl` (grava JSONL no Volume), `list_landing` (lista arquivos) e `make_order` (gera um pedido fictício com campos controlados).

In [0]:
def write_jsonl(path, rows, overwrite=True):
    content = "\n".join(json.dumps(r, ensure_ascii=False) for r in rows)
    dbutils.fs.put(path, content, overwrite=overwrite)
    print(f"Arquivo gravado: {path} | linhas: {len(rows)}")

def list_landing(path=source_path):
    return dbutils.fs.ls(path)

def make_order(order_id, customer_id, product_id, quantity, amount, minutes_offset, channel=None, status="novo"):
    base_time = datetime(2026, 9, 17, 14, 0, 0)
    row = {
        "pedido_id": str(order_id),
        "cliente_id": customer_id,
        "produto_id": product_id,
        "quantidade": int(quantity),
        "valor_total": float(amount),
        "data_pedido": (base_time + timedelta(minutes=minutes_offset)).isoformat(),
        "status": status
    }
    if channel is not None:
        row["canal_venda"] = channel
    return row

## Batch 1 — Geração do primeiro lote de pedidos

Cria 5 pedidos (1 com `pedido_id='null'` e quantidade negativa de propósito para testar a Silver depois).

In [0]:
orders_batch_001 = [
    make_order(1001, "C001", "P100", 2, 59.90, 15, status="novo"),
    make_order(1002, "C002", "P200", 1, 89.50, 16, status="novo"),
    make_order(1003, "C003", "P300", 3, 149.70, 18, status="novo"),
    make_order(1004, "C001", "P400", 1, 39.90, 20, status="novo"),
    make_order('null', "C003", "P600", -5, 99.90, 20, status="novo"),
]

write_jsonl(f"{source_path}/batch_001.json", orders_batch_001)

## Full Load — Carga completa (overwrite)

Lê todos os JSON da pasta e sobrescreve a tabela a cada execução. Simples, mas recarrega tudo sempre — não é incremental.

In [0]:
full_df = (spark.read
    .format("json")
    .load(source_path)
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp()))

(full_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_full))

spark.sql(f"SELECT COUNT(*) AS qtd_linhas FROM {table_full}").show()

## COPY INTO — Carga incremental idempotente

Cria a tabela com schema fixo e usa `COPY INTO`, que rastreia quais arquivos já processou. Reexecutar sem arquivos novos não duplica linhas.

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {table_copy} (
        pedido_id STRING,
        cliente_id STRING,
        produto_id STRING,
        quantidade BIGINT,
        valor_total DOUBLE,
        data_pedido STRING,
        status STRING,
        canal_venda STRING,
        _rescued_data STRING
    ) USING DELTA
""")

spark.sql(f"""
  COPY INTO {table_copy}
  FROM '{source_path}'
  FILEFORMAT = JSON
  FORMAT_OPTIONS ('multiLine' = 'false')
  COPY_OPTIONS ('mergeSchema' = 'true')
""")

spark.sql(f"SELECT COUNT(*) AS qtd_linhas FROM {table_copy}").show()

## Batch 2 — Lote incremental

Gera 3 pedidos novos (1 com quantidade negativa para testar a Silver). Simula chegada de novos arquivos horas depois.

In [0]:
orders_batch_002 = [
    make_order(1005, "C004", "P100", 1, 29.95, 25, status="novo", channel="site"),
    make_order(1006, "C005", "P500", 4, 219.60, 31, status="novo", channel="site"),
    make_order(999, "C007", "P700", -5, 19.00, 23, status="novo", channel="whatsapp"),
]

write_jsonl(f"{source_path}/batch_002_incremental.json", orders_batch_002)

## COPY INTO — Carga incremental

Reexecuta o `COPY INTO`. Como o checkpoint rastreia os arquivos já processados, apenas `batch_002` será carregado.

In [0]:
spark.sql(f"""
    COPY INTO {table_copy}
    FROM '{source_path}'
    FILEFORMAT = JSON
    FORMAT_OPTIONS ('multiLine' = 'false')
    COPY_OPTIONS ('mergeSchema' = 'true')
""")

spark.sql(f"SELECT COUNT(*) AS qtd_linhas FROM {table_copy}").show()

## Auto Loader — Ingestão contínua com checkpoint e schema hints

Configura o Auto Loader com `cloudFiles`, schema hints para tipos corretos e `availableNow=True` para processamento em lote. O checkpoint garante que arquivos já processados não sejam relidos.

In [0]:
schema_hints = (
    "pedido_id STRING, cliente_id STRING, produto_id STRING, "
    "quantidade BIGINT, valor_total DOUBLE, data_pedido STRING, "
    "status STRING, canal_venda STRING"
)

col_order = [
    "pedido_id", "cliente_id", "produto_id",
    "quantidade", "valor_total", "data_pedido",
    "status", "canal_venda", "_rescued_data",
    "_source_file", "_ingested_at"
]

query = (spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "json")
            .option("cloudFiles.schemaLocation", schema_path)
            .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
            .option("cloudFiles.schemaHints", schema_hints)
            .load(source_path)
            .withColumn("_source_file", col("_metadata.file_path"))
            .withColumn("_ingested_at", current_timestamp())
            .select(*col_order)
            .writeStream
            .option("checkpointLocation", checkpoint_path)
            .option("mergeSchema", "true")
            .trigger(availableNow=True)
            .toTable(table_auto)
)
query.awaitTermination()

spark.sql(f"SELECT COUNT(*) AS qtd_linhas FROM {table_auto}").show()

## Schema drift — Novo lote com coluna extra

Gera 2 pedidos com um campo novo (`cupom_desconto`) que não existia antes. Testa a evolução de schema do Auto Loader.

In [0]:
orders_batch_003_schema = [
    {**make_order(1007, "C006", "P600", 2, 179.80, 42, channel="app", status="novo"), "cupom_desconto": "APP10"},
    {**make_order(1008, "C007", "P700", 1, 299.90, 45, channel="web", status="novo"), "cupom_desconto": "WEB20"},
]

write_jsonl(f"{source_path}/batch_003_schema_drift.json", orders_batch_003_schema)

## Auto Loader — Reexecução com evolução de schema

Reexecuta o Auto Loader. A 1ª execução pode falhar (o Auto Loader detecta a mudança de schema); a 2ª execução deve funcionar com a nova coluna adicionada.

In [0]:
query = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(source_path)
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
    .writeStream
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(table_auto))

query.awaitTermination()
spark.sql(f"DESCRIBE TABLE {table_auto}").show(truncate=False)

## CDC — Change Data Capture

Gera eventos de operação (Insert, Update, Delete) sobre pedidos e os carrega na Bronze como histórico cru. As regras de negócio (aplicar o estado final) ficam para o Dia 5.

In [0]:
cdc_path = f"{source_path}_cdc"
dbutils.fs.mkdirs(cdc_path)

cdc_events = [
    {"pedido_id":"2001","cliente_id":"C010","produto_id":"P900","quantidade":1,"valor_total":49.90,"data_pedido":"2026-07-27T12:00:00","op":"I","op_ts":"2026-07-27T12:01:00"},
    {"pedido_id":"2001","cliente_id":"C010","produto_id":"P900","quantidade":2,"valor_total":99.80,"data_pedido":"2026-07-27T12:00:00","op":"U","op_ts":"2026-07-27T12:05:00"},
    {"pedido_id":"2002","cliente_id":"C011","produto_id":"P901","quantidade":1,"valor_total":19.90,"data_pedido":"2026-07-27T12:02:00","op":"I","op_ts":"2026-07-27T12:03:00"},
    {"pedido_id":"2002","cliente_id":"C011","produto_id":"P901","quantidade":1,"valor_total":19.90,"data_pedido":"2026-07-27T12:02:00","op":"D","op_ts":"2026-07-27T12:08:00"},
]

write_jsonl(f"{cdc_path}/cdc_batch_001.json", cdc_events)

cdc_df = (spark.read
    .format("json")
    .load(cdc_path)
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

(cdc_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_cdc_raw))

spark.sql(f"SELECT op, COUNT(*) AS qtd FROM {table_cdc_raw} GROUP BY op ORDER BY op").show()